# Fetch the 58 labeled RSNA-Knee studies (in-kernel, no API listing)

Runs against the competition data mounted at `/kaggle/input/...` — no
`competition_list_files` paging, so no rate limiting. Ports the
`train.csv`/`train_series.csv` narrowing logic from
`scripts/_fetch_rsna_labeled_subset.py` (that part was already correct;
only the *discovery* mechanism changes here) and writes the same
`train_series/<StudyInstanceUID>/<SeriesInstanceUID>/<SOPInstanceUID>.dcm`
layout `RSNAKneeDataset` expects into `/kaggle/working/train_series/`.

After this runs, **Save Version** and pull the output down locally via
`kaggle kernels output <user>/<slug> -p train_series/` or the web UI's
Output tab — no need to touch the local repo's fetch script.

In [ ]:
def find_competition_root(input_root: Path) -> Path:
    """Locates the mounted competition data dir — the one with both
    train.csv and a train_series/ subdir — rather than hardcoding the
    input folder name, since Kaggle mounts a competition under its
    dataset slug, which doesn't always match the competition ref
    (`rsna-knee-abnormality-detection`).

    Checks each direct child of input_root, and also one level deeper
    under `<child>/competitions/*/`, since a competition-list mount nests
    the data there while a single-competition-attach mount puts it
    directly under the child — search both, don't assume either layout."""

    def has_data(path: Path) -> bool:
        return (path / "train.csv").exists() and (path / "train_series").is_dir()

    for candidate in sorted(input_root.iterdir()):
        if has_data(candidate):
            return candidate
        competitions_dir = candidate / "competitions"
        if competitions_dir.is_dir():
            for nested in sorted(competitions_dir.iterdir()):
                if has_data(nested):
                    return nested
    raise FileNotFoundError(
        f"no directory under {input_root} (including one level under any "
        "competitions/ subfolder) has both train.csv and train_series/ — "
        "did you attach the rsna-knee-abnormality-detection competition data?"
    )


COMPETITION_ROOT = find_competition_root(INPUT_ROOT)
print(f"competition data: {COMPETITION_ROOT}")

In [ ]:
# Same narrowing logic as scripts/_fetch_rsna_labeled_subset.py main() —
# only the source paths changed (mounted input, not the repo root).
train = pd.read_csv(COMPETITION_ROOT / "train.csv")
labeled = train.loc[train[TARGET_COLUMNS].notna().any(axis=1), "StudyInstanceUID"].astype(str)
target_studies = set(labeled)
print(f"target: {len(target_studies)} labeled studies")

series_meta = pd.read_csv(COMPETITION_ROOT / "train_series.csv")
series_meta["StudyInstanceUID"] = series_meta["StudyInstanceUID"].astype(str)
series_meta["SeriesInstanceUID"] = series_meta["SeriesInstanceUID"].astype(str)

study_series_planes: dict[str, dict[str, str]] = {}
for study_uid, group in series_meta[series_meta["StudyInstanceUID"].isin(target_studies)].groupby("StudyInstanceUID"):
    study_series_planes[study_uid] = dict(zip(group["SeriesInstanceUID"], group["Anatomical_Plane"]))

total_known_series = sum(len(v) for v in study_series_planes.values())
print(f"these studies have {total_known_series} known series total (across all planes)")

In [ ]:
# Local filesystem walk instead of api.competition_list_files paging —
# the whole reason this notebook exists: the mount already has every
# individual .dcm filename on disk, for free, no rate limit.
n_files_total = 0
n_studies_empty = 0
n_series_missing = 0

for study_uid in sorted(target_studies):
    known_series = study_series_planes.get(study_uid, {})
    study_src = COMPETITION_ROOT / "train_series" / study_uid
    n_files_this_study = 0
    n_series_this_study = 0

    for series_uid in known_series:
        series_src = study_src / series_uid
        if not series_src.is_dir():
            n_series_missing += 1
            continue

        sop_files = sorted(p.name for p in series_src.iterdir() if p.is_file())[:MAX_SLICES_PER_SERIES]
        if not sop_files:
            continue

        series_dst = OUT_ROOT / study_uid / series_uid
        series_dst.mkdir(parents=True, exist_ok=True)
        for sop_file in sop_files:
            dst_file = series_dst / sop_file
            if not dst_file.exists():
                shutil.copyfile(series_src / sop_file, dst_file)
        n_files_this_study += len(sop_files)
        n_series_this_study += 1

    n_files_total += n_files_this_study
    flag = "" if n_files_this_study else "  <-- EMPTY, check this study"
    print(
        f"{study_uid}: {n_series_this_study}/{len(known_series)} series, "
        f"{n_files_this_study} files copied{flag}"
    )
    if not n_files_this_study:
        n_studies_empty += 1

print()
print(
    f"DONE: {len(target_studies) - n_studies_empty}/{len(target_studies)} studies with >=1 file, "
    f"{n_files_total} files copied, {n_series_missing} known series missing from disk, "
    f"output -> {OUT_ROOT}"
)
if n_studies_empty:
    print(f"WARNING: {n_studies_empty} studies came up completely empty — inspect before downloading.")

In [ ]:
# Sanity-check total output size against the ~20GB /kaggle/working quota
# before Save Version.
total_bytes = sum(p.stat().st_size for p in OUT_ROOT.rglob("*") if p.is_file())
print(f"output size: {total_bytes / 1e9:.2f} GB across {sum(1 for _ in OUT_ROOT.rglob('*.dcm'))} files")